In [1]:
# STEP 1: MODEL TRAINING AND PERSISTENCE
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from datetime import datetime, timedelta

In [4]:
# 1. LOAD AND PREPARE DATA (from your notebook)
# ============================================================================

print("\n1. Loading dataset...")

# Load the dataset
df = pd.read_csv("C://Users//Raj03//OneDrive//Desktop//Rajesh_UC_Berkley//kraftwerk//Saravwerk//KaggleV2-May-2016.csv")

print(f"Dataset loaded: {len(df):,} records")

# Data preparation 
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df['ScheduledDate'] = df['ScheduledDay'].dt.date
df['AppointmentDate'] = df['AppointmentDay'].dt.date

# Calculate expected shows
df['ExpectedShow'] = 1 - df['No-show'].map({'Yes': 1, 'No': 0})

# Create daily aggregates
daily_volumes = df.groupby('AppointmentDate').agg({
    'ExpectedShow': 'sum',
    'PatientId': 'count'
}).rename(columns={'PatientId': 'TotalAppointments'})

daily_volumes.index = pd.to_datetime(daily_volumes.index)
daily_volumes = daily_volumes.sort_index()

print(f"Daily aggregates created: {len(daily_volumes)} days")


1. Loading dataset...
Dataset loaded: 110,527 records
Daily aggregates created: 27 days


In [5]:
# 2. FEATURE ENGINEERING 
# ============================================================================

print("\n2. Engineering features...")

# Temporal features
daily_volumes['DayOfWeek'] = daily_volumes.index.dayofweek
daily_volumes['DayOfMonth'] = daily_volumes.index.day
daily_volumes['WeekOfYear'] = daily_volumes.index.isocalendar().week
daily_volumes['Month'] = daily_volumes.index.month
daily_volumes['IsWeekend'] = (daily_volumes['DayOfWeek'] >= 5).astype(int)

# Lag features
for lag in [1, 2, 3, 7, 14]:
    daily_volumes[f'Lag_{lag}'] = daily_volumes['ExpectedShow'].shift(lag)

# Rolling statistics
for window in [3, 7, 14]:
    daily_volumes[f'Rolling_Mean_{window}'] = daily_volumes['ExpectedShow'].rolling(window=window).mean()
    daily_volumes[f'Rolling_Std_{window}'] = daily_volumes['ExpectedShow'].rolling(window=window).std()

# Day-of-week encoding
for day in range(7):
    daily_volumes[f'Day_{day}'] = (daily_volumes['DayOfWeek'] == day).astype(int)

# Drop rows with NaN from lag/rolling features
daily_volumes = daily_volumes.dropna()

print(f"Features engineered: {daily_volumes.shape[1]} total features")


2. Engineering features...
Features engineered: 25 total features


In [6]:
# 3. PREPARE TRAINING DATA
# ============================================================================

print("\n3. Preparing training data...")

# Feature list (same as notebook)
feature_columns = [col for col in daily_volumes.columns if col != 'ExpectedShow' and col != 'TotalAppointments']

X = daily_volumes[feature_columns]
y = daily_volumes['ExpectedShow']

print(f"Features: {len(feature_columns)}")
print(f"Training samples: {len(X)}")


3. Preparing training data...
Features: 23
Training samples: 13


In [7]:
# ============================================================================
# 4. TRAIN FINAL MODEL (Ridge with alpha=1)
# ============================================================================

print("\n4. Training Ridge Regression model (α=1)...")

# Create pipeline (same as notebook)
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1, random_state=42))
])

# Train on ALL available data (for deployment)
ridge_pipeline.fit(X, y)

print(" Model training complete")


4. Training Ridge Regression model (α=1)...
✓ Model training complete


In [8]:
# 5. SAVE MODEL AND FEATURE INFO
# ============================================================================

print("\n5. Saving model and metadata...")

# Save the trained pipeline
joblib.dump(ridge_pipeline, 'hospital_staffing_model.pkl')
print(" Model saved: hospital_staffing_model.pkl")

# Save feature column names (critical for prediction)
joblib.dump(feature_columns, 'feature_columns.pkl')
print(" Feature columns saved: feature_columns.pkl")

# Save metadata
metadata = {
    'model_type': 'Ridge Regression',
    'alpha': 1,
    'n_features': len(feature_columns),
    'training_samples': len(X),
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'expected_mape': 5.35,
    'feature_columns': feature_columns
}

joblib.dump(metadata, 'model_metadata.pkl')
print(" Metadata saved: model_metadata.pkl")



5. Saving model and metadata...
✓ Model saved: hospital_staffing_model.pkl
✓ Feature columns saved: feature_columns.pkl
✓ Metadata saved: model_metadata.pkl


In [9]:
# 6. TEST THE SAVED MODEL
# ============================================================================

print("\n6. Testing saved model...")

# Load the model
loaded_model = joblib.load('hospital_staffing_model.pkl')
loaded_features = joblib.load('feature_columns.pkl')
loaded_metadata = joblib.load('model_metadata.pkl')

# Test prediction on last sample
test_sample = X.iloc[-1:].copy()
prediction = loaded_model.predict(test_sample)

print(f" Test prediction successful: {prediction[0]:.1f} patients")


6. Testing saved model...
✓ Test prediction successful: 3709.5 patients


In [10]:
# 7. CREATE SAMPLE DATA FOR STREAMLIT DEMO
# ============================================================================

print("\n7. Creating sample data for demo...")

# Save last 30 days of actual data for comparison
recent_data = daily_volumes.tail(30)[['ExpectedShow']].copy()
recent_data.to_csv('recent_actual_volumes.csv')
print(" Sample data saved: recent_actual_volumes.csv")


7. Creating sample data for demo...
✓ Sample data saved: recent_actual_volumes.csv
